[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/02_error_propagation_and_stability_tricks/exercises.ipynb)

# Exercises — Topic 02: Error Propagation and Stability Tricks

20 fully solved problems in 4 levels. Attempt each before opening its solution cell.

## Level 0 — Concept Check

### Problem L0.1: Where cancellation's error comes from

True or false, with justification: "When two nearly equal floats are subtracted, the subtraction operation itself commits a large rounding error."

**Solution**

**False.** By the Sterbenz lemma, if $y/2 \le x \le 2y$ the difference $x - y$ is *exactly* representable and the subtraction commits zero error. The catastrophe is different: the leading digits — which agreed and carried the certified information — cancel away, so whatever relative errors $\delta_x, \delta_y$ the operands carried *from earlier steps* now constitute the leading digits of the small result. The amplification factor is

$$
\frac{x + y}{\lvert x - y \rvert} \gg 1
$$

$$
\boxed{\text{False — cancellation amplifies pre-existing error; the subtraction itself is often exact.}}
$$

*Key takeaway*: fix cancellation upstream (restructure the formula), not at the subtraction.

### Problem L0.2: Forward vs backward error

An algorithm for $y = f(x) = x^2$ returns $\hat{y} = 4.2$ for input $x = 2$. Compute (a) the relative forward error and (b) the relative backward error. Which one certifies the *algorithm* rather than the *problem*?

**Solution**

**(a)** Exact value $y = 4$:

$$
\text{forward error} = \frac{\lvert 4.2 - 4 \rvert}{4} = 0.05
$$

**(b)** Find $\Delta x$ with $(x + \Delta x)^2 = 4.2$: $x + \Delta x = \sqrt{4.2} \approx 2.0494$, so

$$
\text{backward error} = \frac{0.0494}{2} \approx 0.0247
$$

Backward error certifies the algorithm: it says "the answer produced is exact for an input 2.5% away." Whether that is acceptable depends on the input's own uncertainty; forward error additionally entangles the problem's conditioning (here $\kappa = 2$ for squaring, and indeed $0.05 \approx 2 \times 0.0247$).

$$
\boxed{\text{forward} = 5\%, \quad \text{backward} \approx 2.5\%, \quad \text{forward} \approx \kappa \times \text{backward}}
$$

*Key takeaway*: backward error is the algorithm's report card; conditioning converts it to forward error.

### Problem L0.3: The digit-loss estimate

Two measured quantities are accurate to 10 significant digits and agree with each other to 7 significant digits. Roughly how many trustworthy digits does their difference carry?

**Solution**

By the cancellation bound, the difference's relative error is amplified by $\frac{x+y}{\lvert x-y \rvert} \approx 2 \times 10^{7}/2 = 10^{7}$ relative to the operands' error $10^{-10}$:

$$
\text{relative error of difference} \approx 10^{-10} \times 10^{7} = 10^{-3}
$$

$$
\boxed{\approx 3 \text{ significant digits } (10 - 7)}
$$

*Key takeaway*: digits of agreement = digits lost. Subtracting quantities that agree to 7 digits deletes 7 digits of certified accuracy.

### Problem L0.4: Pick the stable expression

For each pair, pick the numerically preferable form for small $x \gt 0$ and name the failure of the other: (a) $1 - \cos x$ vs $2\sin^2(x/2)$; (b) $\log(1+x)$ vs `log1p(x)`; (c) $\sqrt{x+1} - \sqrt{x}$ vs $1/(\sqrt{x+1} + \sqrt{x})$ for large $x$.

**Solution**

**(a)** $2\sin^2(x/2)$. For small $x$, $\cos x \approx 1 - x^2/2$ agrees with 1 to many digits, so $1 - \cos x$ cancels catastrophically (for $x = 10^{-8}$ it returns 0 instead of $5 \times 10^{-17}$). The half-angle form squares a well-conditioned quantity — no subtraction at all.

**(b)** `log1p(x)`. Forming $\mathrm{fl}(1 + x)$ truncates $x$ to the $\varepsilon$-grid around 1, destroying up to all of $x$'s digits when $x \lt \varepsilon$.

**(c)** $1/(\sqrt{x+1} + \sqrt{x})$. For large $x$ the two roots agree to $\sim \log_{10}\sqrt{x}$ digits and the difference cancels; the reciprocal-sum form adds positives.

$$
\boxed{\text{(a) } 2\sin^2(x/2), \quad \text{(b) log1p}, \quad \text{(c) } 1/(\sqrt{x+1}+\sqrt{x})}
$$

*Key takeaway*: each stable form performs the dangerous subtraction symbolically, where algebra is exact.

## Level 1 — Foundation

### Problem L1.1: Applying the $\gamma_n$ bound

A dot product $s = \sum_{i=1}^{n} a_i b_i$ of vectors with $n = 10^{6}$ is computed naively in binary64. (a) State the standard backward-error result. (b) Give the worst-case forward relative error bound when all products are positive.

**Solution**

**(a)** The computed value satisfies (Higham, Sec. 3.1)

$$
\hat{s} = \sum_{i=1}^{n} a_i b_i (1 + \theta_i), \qquad \lvert \theta_i \rvert \le \gamma_n = \frac{nu}{1 - nu}
$$

i.e. the exact dot product of elementwise-perturbed inputs — the algorithm is backward stable.

**(b)** For same-signed products, $\lvert \hat{s} - s \rvert \le \gamma_n \sum \lvert a_i b_i \rvert = \gamma_n \lvert s \rvert$, so

$$
\frac{\lvert \hat{s} - s \rvert}{\lvert s \rvert} \le \gamma_{10^{6}} \approx 10^{6} \times 1.11 \times 10^{-16} \approx 1.1 \times 10^{-10}
$$

$$
\boxed{\text{relative error} \le \gamma_n \approx nu \approx 1.1 \times 10^{-10}}
$$

*Key takeaway*: a million-term positive dot product still guarantees ~10 digits — worst case. Mixed signs transfer the bound to $\sum \lvert a_i b_i \rvert$, and relative accuracy then depends on cancellation in $s$ itself.

### Problem L1.2: Summation order matters

Sum $x = (10^{16}, 1, 1, \dots, 1)$ with $10^{4}$ ones, in binary64. Compute the result of (a) left-to-right summation starting at $10^{16}$, (b) summing the ones first. Exact answer: $10^{16} + 10^{4}$.

**Solution**

**(a)** After $s = 10^{16}$, each `s + 1` must round: the spacing at $10^{16}$ is $\mathrm{ulp} = 2$ (since $2^{53} \approx 9.01 \times 10^{15} \le 10^{16} \lt 2^{54}$, gap $= 2^{54-52}$... more precisely $10^{16} \in [2^{53}, 2^{54})$ so gap $= 2$). Adding 1 gives a tie at midpoint; ties-to-even rounds back to the even multiple — the sum *never moves*:

$$
\hat{S}_a = 10^{16}
$$

Absolute error $10^{4}$.

**(b)** The $10^{4}$ ones sum exactly to $10^{4}$ (integers below $2^{53}$). Then one addition:

$$
\hat{S}_b = \mathrm{fl}(10^{16} + 10^{4}) = 10^{16} + 10^{4}
$$

exactly, since $10^{16} + 10^{4}$ is an even integer below $2^{54}$, hence representable. Zero error.

$$
\boxed{\hat{S}_a = 10^{16} \; (\text{error } 10^{4}), \qquad \hat{S}_b = 10^{16} + 10^{4} \; (\text{exact})}
$$

*Key takeaway*: sum small terms first so they can pile up above the big term's ulp before being absorbed.

### Problem L1.3: Stable quadratic formula in action

Solve $x^2 - 10^{8} x + 1 = 0$ in binary64: (a) show what the naive formula gives for the small root, (b) compute both roots stably.

**Solution**

**(a)** Here $b = -10^{8}$, and $\sqrt{b^2 - 4ac} = \sqrt{10^{16} - 4}$. In binary64, $\mathrm{fl}(10^{16} - 4) = 10^{16} - 4$ but the square root $\approx 10^{8}(1 - 2 \times 10^{-16})$ rounds to $\approx 99999999.99999998$. The naive small root

$$
x_2 = \frac{10^{8} - \sqrt{10^{16} - 4}}{2}
$$

subtracts two values agreeing to ~16 digits: the computed result is built entirely from the last one or two bits — accurate to maybe 1 digit (typically $\approx 7.45 \times 10^{-9}$ instead of $10^{-8}$).

**(b)** Stable route: $\operatorname{sign}(b) = -1$, so

$$
q = -\tfrac{1}{2}\left(b - \sqrt{b^2 - 4ac}\right) = \tfrac{1}{2}\left(10^{8} + \sqrt{10^{16} - 4}\right) \approx 10^{8}
$$

$$
x_1 = \frac{q}{a} \approx 1.0 \times 10^{8}, \qquad x_2 = \frac{c}{q} = \frac{1}{10^{8}} = 1.0 \times 10^{-8}
$$

both to full precision (check: $x_1 x_2 = c/a = 1$ ✓).

$$
\boxed{x_1 \approx 10^{8}, \quad x_2 = 10^{-8} \text{ via } x_2 = c/q \text{ (naive form loses ~15 digits)}}
$$

*Key takeaway*: Vieta's identities are exact algebra — use them to dodge every cancelling branch of a formula.

### Problem L1.4: `expm1` error estimate

For $x = 10^{-10}$ in binary64, estimate the relative error of computing $e^{x} - 1$ (a) naively and (b) via `expm1`. Assume `exp` is faithful to 1 ulp.

**Solution**

**(a)** $e^{x} = 1 + 10^{-10} + \tfrac{1}{2}10^{-20} + \cdots$. The computed $\mathrm{fl}(e^{x})$ lies on the $\varepsilon$-grid near 1: absolute quantization $\le \tfrac{1}{2}\varepsilon = 1.1 \times 10^{-16}$. Subtracting 1 is then exact (Sterbenz), but the retained error relative to the true answer $\approx 10^{-10}$ is

$$
\frac{1.1 \times 10^{-16}}{10^{-10}} \approx 1.1 \times 10^{-6}
$$

— about 6 digits lost.

**(b)** `expm1` evaluates the series in terms of $x$ directly, returning $(e^{x} - 1)(1 + \delta)$ with $\lvert \delta \rvert \lesssim u \approx 1.1 \times 10^{-16}$: full precision.

$$
\boxed{\text{naive: rel. err.} \sim 10^{-6}; \quad \texttt{expm1}: \text{rel. err.} \sim 10^{-16}}
$$

*Key takeaway*: the digit loss equals $\log_{10}(\varepsilon/x)$ — the smaller the signal, the more digits the detour through 1 destroys.

### Problem L1.5: `hypot` and overflow

Let $a = 3 \times 10^{200}$, $b = 4 \times 10^{200}$ (binary64, max $\approx 1.8 \times 10^{308}$). (a) Show the naive $\sqrt{a^2 + b^2}$ fails. (b) Derive the scaled algorithm and its result.

**Solution**

**(a)** $a^2 = 9 \times 10^{400} \gt 1.8 \times 10^{308}$: the square overflows to $\infty$, and $\sqrt{\infty} = \infty$. The true answer $5 \times 10^{200}$ is perfectly representable — the *algorithm*, not the *problem*, overflows.

**(b)** Factor out $M = \max(\lvert a \rvert, \lvert b \rvert) = 4 \times 10^{200}$:

$$
\sqrt{a^2 + b^2} = M \sqrt{1 + \left(\frac{\min}{M}\right)^2} = 4 \times 10^{200} \sqrt{1 + 0.5625} = 4 \times 10^{200} \times 1.25 = 5 \times 10^{200}
$$

All intermediates lie in $[1, 2]$ — no overflow or underflow possible for any representable inputs.

$$
\boxed{\texttt{hypot}(a,b) = 5 \times 10^{200} \text{ via } M\sqrt{1 + (m/M)^2}}
$$

*Key takeaway*: rescale by the dominant magnitude before squaring — the same trick that stabilizes vector norms, Givens rotations, and softmax.

### Problem L1.6: Variance formulas head-to-head

Data: $x = (10^{6} + 1,\; 10^{6} + 2,\; 10^{6} + 3)$ in binary32 ($u \approx 6 \times 10^{-8}$). Exact variance (population) is $2/3$. Analyze what (a) the one-pass formula $\overline{x^2} - \bar{x}^2$ and (b) the shifted/two-pass formula produce.

**Solution**

**(a)** $\overline{x^2} \approx 10^{12}$, $\bar{x}^2 \approx 10^{12}$. binary32 carries $\approx 7.2$ digits, i.e. absolute resolution at $10^{12}$ of $\mathrm{ulp} \approx 10^{12} \times 1.2 \times 10^{-7} \approx 1.2 \times 10^{5}$. The true difference $2/3$ is *five orders of magnitude below one ulp* of the operands: the subtraction returns rounding garbage — any multiple of $\sim 10^{5}$, possibly negative.

**(b)** Shift by $c = 10^{6}$ first: residuals $(1, 2, 3)$, mean $2$, squared deviations $(1, 0, 1)$:

$$
\mathrm{Var} = \frac{1 + 0 + 1}{3} = \frac{2}{3}
$$

computed exactly even in binary32, since all intermediates are tiny integers.

$$
\boxed{\text{one-pass: noise of order } 10^{5} \text{ (can be negative); shifted two-pass: exactly } 2/3}
$$

*Key takeaway*: variance is a property of *residuals*; any algorithm that touches raw second moments inherits a $\mu^2/\sigma^2$ amplification factor.

## Level 2 — Applications in AI/ML

### Problem L2.1: fp16 loss accumulation stalls

An epoch accumulates 50000 batch losses, each $\approx 2.0$, into an fp16 scalar. (a) At what accumulated value does adding 2.0 stop changing the sum? (b) After how many batches is that reached, and what is the reported epoch mean vs the true mean?

**Solution**

**(a)** Adding 2.0 changes the sum as long as $2.0 \ge \tfrac{1}{2}\mathrm{ulp}(s)$... more precisely the addition rounds to $s$ itself once $\mathrm{ulp}(s) \gt 4.0$ (then $s + 2$ lies below the midpoint). fp16 spacing is $2^{e-10}$; $\mathrm{ulp} = 4$ when $2^{e-10} = 4$, i.e. $e = 12$: for $s \ge 2^{12} = 4096$ the spacing is 4 and $s + 2$ is a tie rounding to even — effectively stalling at

$$
s^{*} = 4096
$$

(around $s \in [2048, 4096)$ spacing is 2 and additions still move the sum.)

**(b)** Stall after $\approx 4096 / 2 = 2048$ batches. Reported epoch total stays $4096$; reported mean $= 4096/50000 \approx 0.082$ versus true mean $2.0$ — a 96% underestimate, silently.

$$
\boxed{\text{stalls at } s \approx 4096 \text{ after } \approx 2048 \text{ batches; reported mean } 0.08 \text{ vs true } 2.0}
$$

*Key takeaway*: never accumulate metrics in half precision — use fp32/fp64 accumulators (a wider accumulator multiplies headroom by $2^{13}$).

### Problem L2.2: Kahan summation step, by hand

Run one Kahan iteration in a toy decimal system with 4 significant digits: current sum $s = 1000$, compensation $c = 0$, incoming $x = 3.141$. Show what is lost by the raw addition and what the compensation captures.

**Solution**

Step by step (4-digit rounding after each op):

1. $y = x - c = 3.141$
2. $t = s + y = \mathrm{fl}(1003.141) = 1003$ — the digits $0.141$ are *lost* by rounding.
3. $c = (t - s) - y = (1003 - 1000) - 3.141 = 3 - 3.141 = -0.141$ — both subtractions are exact (small integers, then 4-digit result), so $c$ records exactly the lost part, with sign convention "amount over-counted".
4. $s = 1003$.

Next iteration subtracts $c$: $y' = x' - (-0.141) = x' + 0.141$ — the lost mass is re-injected. The pair satisfies $s - c = 1003 - (-0.141) = 1003.141$, the exact sum.

$$
\boxed{t = 1003, \quad c = -0.141, \quad s - c = 1003.141 \text{ (exact partial sum preserved)}}
$$

*Key takeaway*: the compensation variable is the exact rounding error of the addition, recovered by Fast2Sum — nothing is heuristic in Kahan's loop.

### Problem L2.3: Numerically stable BCE loss

The binary cross-entropy $\ell = -[y \log \sigma(z) + (1-y)\log(1 - \sigma(z))]$ with $\sigma(z) = 1/(1+e^{-z})$ fails for large $\lvert z \rvert$. (a) Exhibit the failure at $z = 40$, $y = 0$ (binary64). (b) Derive the stable "logits" form used by frameworks.

**Solution**

**(a)** $\sigma(40) = 1/(1 + e^{-40})$; since $e^{-40} \approx 4.2 \times 10^{-18} \lt \varepsilon$, $\mathrm{fl}(\sigma(40)) = 1.0$ exactly. Then $\log(1 - \sigma) = \log(0) = -\infty$: the loss overflows although the true value is merely $\approx 40$.

**(b)** Substitute and simplify symbolically:

$$
\ell = -y \log \sigma(z) - (1-y) \log(1 - \sigma(z)) = (1 - y) z + \log(1 + e^{-z})
$$

valid for $z \gt 0$; using $\log(1+e^{-\lvert z \rvert})$ and rearranging to cover both signs:

$$
\ell = \max(z, 0) - yz + \mathrm{log1p}(e^{-\lvert z \rvert})
$$

Every term is bounded and cancellation-free: for $z = 40, y = 0$, $\ell = 40 + \mathrm{log1p}(4.2 \times 10^{-18}) = 40 + 4.2 \times 10^{-18}$ — exact to machine precision.

$$
\boxed{\ell = \max(z, 0) - yz + \mathrm{log1p}(e^{-\lvert z \rvert})}
$$

*Key takeaway*: this is `BCEWithLogitsLoss`; the fusion of sigmoid and log is not an optimization but a correctness requirement.

### Problem L2.4: Welford in a BatchNorm running-stat loop

A BatchNorm layer tracks running variance of activations with mean $\mu \approx 50$ and std $\sigma \approx 0.1$ in float32. Estimate the relative error of the one-pass $\overline{x^2} - \bar{x}^2$ estimate, and explain why Welford/two-pass is required.

**Solution**

Amplification factor of the cancelling subtraction: the two terms are $\approx \mu^2 = 2500$ while the result is $\sigma^2 = 0.01$ — a ratio of

$$
\frac{\mu^2}{\sigma^2} = \frac{2500}{0.01} = 2.5 \times 10^{5}
$$

Each term carries relative error $\gtrsim u_{32} \approx 6 \times 10^{-8}$ (more after summation over the batch), so the variance estimate has relative error at least

$$
2.5 \times 10^{5} \times 6 \times 10^{-8} \approx 1.5\%
$$

and summation error over $n$ terms multiplies this further ($\sqrt{n}$ to $n$ times). A noise floor above 1.5% on $\sigma^2$ feeds directly into the normalization scale $1/\sqrt{\sigma^2 + \epsilon}$ every forward pass. Welford's update subtracts only residuals $x - m$ of size $\sigma$, so its error stays $O(u)$ *relative to $\sigma^2$* — about six orders of magnitude better here.

$$
\boxed{\text{one-pass rel. error} \gtrsim \frac{\mu^2}{\sigma^2} u \approx 1.5\%; \quad \text{Welford keeps } O(u)}
$$

*Key takeaway*: the amplification $\mu^2/\sigma^2$ tells you in advance when the textbook variance formula is unusable — check it before trusting any streaming statistic.

### Problem L2.5: Averaging gradients across workers

In data-parallel training, $W = 1024$ workers each produce a gradient tensor; the all-reduce computes $\bar{g} = \frac{1}{W}\sum_w g_w$ in fp16, where per-worker gradient components are $O(10^{-3})$ with random signs. (a) Bound the naive sequential-reduction error. (b) Explain why tree (pairwise) all-reduce plus fp32 accumulation is the standard.

**Solution**

**(a)** Sequential reduction over $W$ terms has error bound $\gamma_{W-1} \sum_w \lvert g_w \rvert$ with $u_{16} = 2^{-11}$:

$$
\gamma_{1023} \approx 1023 \times 4.9 \times 10^{-4} \approx 0.5
$$

— the bound is a **50% relative perturbation** of the sum of magnitudes. With random signs the true mean is $\sim \sqrt{W}$ smaller than $\sum \lvert g_w \rvert$, making the worst-case bound larger than the signal itself: the naive fp16 reduction can be pure noise.

**(b)** A binary-tree reduction has depth $\log_2 W = 10$, replacing the factor 1023 by 10: bound $\approx 10 \times 4.9 \times 10^{-4} \approx 0.5\%$. Accumulating in fp32 ($u = 6 \times 10^{-8}$) drops even the sequential bound to $\approx 6 \times 10^{-5}$. Combining both — tree topology (which is also communication-optimal) with fp32 accumulators — makes reduction error negligible relative to gradient noise.

$$
\boxed{\text{naive fp16: } \gamma_{1023} \approx 0.5 \text{ (useless); tree + fp32: } \lesssim 10^{-6}}
$$

*Key takeaway*: reduction *topology* is an accuracy decision, not just a bandwidth one — depth, not width, drives the error bound.

### Problem L2.6: Finite-difference gradient checking

Gradient-checking a scalar loss $f$ in binary64 uses the central difference $D_h = \frac{f(x+h) - f(x-h)}{2h}$. (a) Write the total error as truncation plus round-off. (b) Derive the optimal $h$ and the best achievable accuracy, taking $f$ and its derivatives $O(1)$.

**Solution**

**(a)** Taylor expansion gives truncation error $\frac{h^2}{6} f'''(\xi) = O(h^2)$. Each evaluation of $f$ carries error $\sim u$, and the difference of two nearly equal $O(1)$ values divided by $2h$ contributes round-off $\sim u/h$:

$$
E(h) \approx c_1 h^2 + c_2 \frac{u}{h}
$$

**(b)** Minimize: $E'(h) = 2c_1 h - c_2 u / h^2 = 0$ gives

$$
h^{*} = \left(\frac{c_2 u}{2 c_1}\right)^{1/3} \sim u^{1/3} \approx (1.1 \times 10^{-16})^{1/3} \approx 5 \times 10^{-6}
$$

with minimal error

$$
E(h^{*}) \sim u^{2/3} \approx 2 \times 10^{-11}
$$

$$
\boxed{h^{*} \sim u^{1/3} \approx 5 \times 10^{-6}, \qquad E_{\min} \sim u^{2/3} \approx 10^{-11}}
$$

*Key takeaway*: this is why gradient checks pass at $10^{-7}$-ish tolerances and *cannot* do better — and why analytic/autodiff gradients (exact to $O(u)$) are used for training itself.

## Level 3 — Challenge

### Problem L3.1: Prove Fast2Sum is error-free

Prove: for floats $a, b$ with $\lvert a \rvert \ge \lvert b \rvert$, the sequence $s = \mathrm{fl}(a+b)$, $z = \mathrm{fl}(s-a)$, $t = \mathrm{fl}(b-z)$ satisfies $s + t = a + b$ exactly (binary arithmetic, round-to-nearest, no overflow).

**Solution**

**Step 1 — $z$ is exact.** Let $s = (a+b)(1+\delta)$. We claim $s - a$ is representable. Since $\lvert a \rvert \ge \lvert b \rvert$, we have $\lvert s \rvert \le 2\lvert a \rvert$ and $s$ has the same sign as $a$ (as $\lvert b \rvert \le \lvert a \rvert$ implies $a + b$ shares $a$'s sign or is 0). Also $\lvert s \rvert \ge \lvert a \rvert / 2$: indeed $\lvert a + b \rvert \ge \lvert a \rvert - \lvert b \rvert$, and if $\lvert b \rvert \le \lvert a \rvert/2$ this gives $\ge \lvert a \rvert/2$ directly; if $\lvert a \rvert/2 \lt \lvert b \rvert \le \lvert a \rvert$, Sterbenz makes $a + b$ exact so $s = a+b$ and $z = b$ trivially, $t = 0$. In the nontrivial case, $a/2 \le s \le 2a$ (up to sign), so by the Sterbenz lemma $s - a$ is exactly representable:

$$
z = s - a \quad \text{exactly}
$$

**Step 2 — $t$ is exact.** $z = s - a$ where $s = \mathrm{fl}(a + b)$, so $b - z = b - (s - a) = (a + b) - s$ — precisely the (negated) rounding error of the first addition. The rounding error of adding two $p$-bit floats is known to be representable in $p$ bits (its magnitude is $\le \tfrac{1}{2}\mathrm{ulp}(s)$ and it is a multiple of $\mathrm{ulp}$ of the smaller operand's last bit — a standard lemma). Hence the subtraction $b - z$ commits no rounding:

$$
t = (a + b) - s \quad \text{exactly}
$$

**Step 3 — conclude.**

$$
s + t = s + (a + b) - s = a + b \qquad \blacksquare
$$

$$
\boxed{s + t = a + b \text{ exactly: 3 flops recover the rounding error of an addition}}
$$

*Key takeaway*: rounding errors of $+$ are themselves floats — the bedrock fact beneath Kahan summation, double-double arithmetic, and exact geometric predicates.

### Problem L3.2: Condition number of summation and a sharpness example

Define $\kappa_{\text{sum}}(x) = \frac{\sum_i \lvert x_i \rvert}{\lvert \sum_i x_i \rvert}$. (a) Show the relative error of any backward-stable summation is bounded by $\approx \kappa_{\text{sum}} \cdot \gamma$, with $\gamma$ the algorithm's coefficient. (b) Construct, for any target $K \ge 1$, a 3-term dataset with $\kappa_{\text{sum}} = K$. (c) What does $\kappa_{\text{sum}}$ predict for mean-centering already-centered data?

**Solution**

**(a)** Backward stability gives $\hat{S} = \sum x_i(1 + \epsilon_i)$, $\lvert \epsilon_i \rvert \le \gamma$. Then

$$
\frac{\lvert \hat{S} - S \rvert}{\lvert S \rvert} \le \frac{\sum \lvert x_i \rvert \lvert \epsilon_i \rvert}{\lvert S \rvert} \le \gamma \cdot \frac{\sum \lvert x_i \rvert}{\lvert S \rvert} = \gamma\, \kappa_{\text{sum}} \qquad \blacksquare
$$

**(b)** Take $x = \left(\frac{K}{2}, -\frac{K}{2} + \frac{1}{2}, \frac{1}{2}\right)$ scaled at will: $\sum x_i = 1$ and $\sum \lvert x_i \rvert = K$ (for $K \ge 1$), so $\kappa_{\text{sum}} = K$ exactly. As $K \to \infty$ the sum is a vanishing residual of large cancelling terms and *no* algorithm computing in working precision can guarantee relative accuracy — the information is not in the rounded inputs.

**(c)** Centered data has $\sum x_i \approx 0$ while $\sum \lvert x_i \rvert$ is large: $\kappa_{\text{sum}} \to \infty$. Predicted: the computed mean of centered data is *absolute*-accurate ($\sim \gamma \sum \lvert x_i \rvert / n$) but has no *relative* accuracy — harmless if later used additively, dangerous if divided by.

$$
\boxed{\text{rel. err.} \le \gamma \kappa_{\text{sum}}, \quad \kappa_{\text{sum}} = \frac{\sum \lvert x_i \rvert}{\lvert \sum x_i \rvert} \text{ arbitrarily large under cancellation}}
$$

*Key takeaway*: separate the algorithm's $\gamma$ (you control it) from the data's $\kappa_{\text{sum}}$ (you don't) — accuracy is always their product.

### Problem L3.3: Error of the sample-mean update rule

Streaming mean via $m_k = m_{k-1} + (x_k - m_{k-1})/k$. (a) Prove this recurrence is algebraically exact for the mean. (b) Explain its numerical superiority over $m_k = \frac{(k-1) m_{k-1} + x_k}{k}$ and over summing-then-dividing, when $n$ is large and data is nearly constant, $x_i = \mu + \eta_i$ with $\lvert \eta_i \rvert \ll \mu$.

**Solution**

**(a)** Induction: assume $m_{k-1} = \frac{1}{k-1}\sum_{i \lt k} x_i$. Then

$$
m_k = m_{k-1} + \frac{x_k - m_{k-1}}{k} = \frac{k m_{k-1} - m_{k-1} + x_k}{k} = \frac{(k-1)m_{k-1} + x_k}{k} = \frac{1}{k}\sum_{i \le k} x_i \qquad \blacksquare
$$

**(b)** Numerically the three forms differ in *what gets rounded*:

- **Residual form**: the update term $(x_k - m_{k-1})/k$ has magnitude $\sim \lvert \eta \rvert / k$ — tiny and *relative to the deviation*. Rounding it perturbs the mean by $O(u \lvert \eta \rvert)$ per step; errors on the $O(\mu)$ base never occur because $\mu$ is never re-manufactured.
- **Weighted form** $((k-1)m + x)/k$: multiplies the $O(\mu)$ quantity by $(k-1)$ and back-divides — two extra roundings of size $u\mu$ *per step*, giving accumulated drift $O(n u \mu)$, worse by the factor $\mu/\lvert \eta \rvert$.
- **Sum-then-divide**: the raw sum reaches $n\mu$; if the accumulator's ulp at $n\mu$ exceeds $\lvert x \rvert$ the sum stalls (Problem L2.1); pairwise/Kahan repair this but the residual form never faces it.

$$
\boxed{m_k = m_{k-1} + \frac{x_k - m_{k-1}}{k} \text{ is exact algebra and rounds only residual-scale quantities}}
$$

*Key takeaway*: update rules that move a state by *residuals* are self-stabilizing — the same structural reason Welford, Kahan, and momentum-style optimizers are robust.

### Problem L3.4: A backward-stable algorithm with terrible forward error

Consider computing $f(x) = 1 - \cos x$ at $x = 1.2 \times 10^{-8}$ in binary64 via the naive expression. (a) Show the computed answer has ~100% forward error. (b) Show the computation is nonetheless *backward stable*, and reconcile with forward err $\approx \kappa \times$ backward err. (c) Give the stable rewrite and its error.

**Solution**

**(a)** True value: $1 - \cos x = \frac{x^2}{2}(1 + O(x^2)) \approx 7.2 \times 10^{-17}$. Computed: $\mathrm{fl}(\cos x)$ is within $u$ of $\cos x = 1 - 7.2 \times 10^{-17}$; since that is within half an ulp of $1$, $\mathrm{fl}(\cos x) = 1$ and the subtraction returns $0$ — forward relative error $= 1$ (100%).

**(b)** Backward view: $0 = 1 - \cos(\tilde{x})$ holds exactly for $\tilde{x} = 0$, and $\frac{\lvert \tilde{x} - x \rvert}{\lvert x \rvert} = 1$... that input perturbation is large. Sharper: the subtraction and cosine each committed $\le u$; attribute the error to $\cos$: $\hat{y} = 1 - \cos(x)(1+\delta)$-type perturbations correspond to input perturbation $\Delta x \approx \delta \cot(x) \cdot$..., but the honest statement is: each *operation* was backward stable, while the *composition* is only "stable in the mixed sense". The reconciliation uses the condition number of $f$ at $x$:

$$
\kappa_f(x) = \left\lvert \frac{x f'(x)}{f(x)} \right\rvert = \frac{x \sin x}{1 - \cos x} \approx \frac{x \cdot x}{x^2/2} = 2
$$

— the *problem* is well-conditioned! The failure is that the naive algorithm is **not** backward stable as a whole: no small input perturbation maps $7.2 \times 10^{-17}$ to $0$ through $f$ with relative input change $O(u)$; the intermediate quantity $\cos x$ absorbed an error that is $O(u)$ *for cosine* but catastrophic *for the difference*. Backward stability is not composable.

**(c)** Rewrite: $1 - \cos x = 2\sin^2(x/2)$. Then $\sin(x/2) \approx 6 \times 10^{-9}$ computed to relative $O(u)$, squared and doubled: total relative error $\lesssim 4u$.

$$
\boxed{\text{naive: } 100\% \text{ error despite per-op stability; } 2\sin^2(x/2) \text{ achieves } O(u) \text{ for a } \kappa \approx 2 \text{ problem}}
$$

*Key takeaway*: stability of every individual operation does not grant stability of the pipeline — analyze (or restructure) the composition, which is the entire subject of Topic 03.